# Phase 3 — Scoped Specification Extraction (DeepSeek)
This notebook uses real API calls, and sends selected SAP excerpts to DeepSeek. Only proceed if authorized. No fixture fallback. Candidate requirements are not clinically confirmed. Same Drive input and targets as Phase 2.1.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'generic-retrieval-v2'
ROOT = Path('/content/tfl-generic-retrieval-v2')
if not ROOT.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)])
else:
    current = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'], text=True).strip()
    assert current == BRANCH, f'Wrong branch: {current}'
    subprocess.check_call(['git','-C',str(ROOT),'pull','--ff-only'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[notebook,test]','sentence-transformers>=3,<6'])
SRC = (ROOT / 'src').resolve()
assert (SRC / 'tfl_agent/tools/sap_hybrid.py').is_file(), SRC
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
cached = sys.modules.get('tfl_agent')
if cached is not None and not Path(cached.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('Wrong tfl_agent checkout already imported. Restart session and run all.')
import tfl_agent
from tfl_agent.agent.sap_scope_pilot import run_scope_retrieval
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC)
print('Commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent package loaded:', tfl_agent.__file__)
from tfl_agent.tools.sap_hybrid import LocalEncoder
from tfl_agent.tools.llm_client import DeepSeekJSONClient
from tfl_agent.agent.scoped_specification import run_specification_extraction

## 1. Load SAP from the existing Google Drive folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAP_RELATIVE_PATH = 'Colab Notebooks/agent/midterm demo/SAP_001_00.pdf'
SAP_PATH = Path('/content/drive/MyDrive') / DRIVE_SAP_RELATIVE_PATH
SAP_BODY_PAGE_RANGE = (4, 21)
TARGETS = [
    {'tfl_id':'14.1.3.1','output_type':'table','title':'Demographic and Baseline Characteristics – Safety Analysis Set'},
    {'tfl_id':'14.2.3.1','output_type':'table','title':'Summary of Progression Free Survival – ITT Analysis Set'},
]
if not SAP_PATH.is_file():
    available = sorted(p.name for p in SAP_PATH.parent.glob('*.pdf'))
    raise FileNotFoundError(f'{SAP_PATH}; PDFs in folder: {available}')
SAVE_TO_DRIVE = True
print('SAP:', SAP_PATH)
print('Physical pages:', SAP_BODY_PAGE_RANGE)

## 2. DeepSeek authorization and configuration
Enable notebook access to DEEPSEEK_API_KEY in Colab Secrets. Confirm below only if you have authorization to send these SAP excerpts to DeepSeek. API calls can incur charges; 3–6 logical extraction requests for two targets, excluding retries.

In [ ]:
from google.colab import userdata
MODEL = 'deepseek-flash'
AUTHORIZED_TO_SEND_SAP = False #@param {type:"boolean"}
if not AUTHORIZED_TO_SEND_SAP:
    raise RuntimeError('Confirm permission to send SAP excerpts by checking AUTHORIZED_TO_SEND_SAP, then rerun this cell.')
key = userdata.get('DEEPSEEK_API_KEY')
if not key:
    raise RuntimeError('Missing DEEPSEEK_API_KEY in Colab Secrets')
llm = DeepSeekJSONClient(model=MODEL, api_key=key)
del key
print('DeepSeek client initialized; key not printed.')

## 3. Rebuild scoped evidence (local embedding inference)
The actual SAP is read from Drive. The same encoder is reused for missing-field retrieval. No previous ZIP upload is required.

In [ ]:
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = Path('/content/SAP_Specification_Phase3') / RUN_ID
RET = OUT / 'Retrieval'
encoder = LocalEncoder()
ret_summary, _ = run_scope_retrieval(SAP_PATH, SAP_BODY_PAGE_RANGE, TARGETS, RET, encoder=encoder, compare_phase2=False)
print(json.dumps(ret_summary, indent=2))

## 4. Extract study and target requirements
Exact quotations, source offsets and scope checks are applied. Missing requirements trigger at most one additional completion call per scope. Raw responses are kept in the audit; do not treat them as confirmed clinical facts.

In [ ]:
summary, study, specs = run_specification_extraction(RET, OUT, llm, encoder=encoder)
print(json.dumps(summary, indent=2))
display(pd.DataFrame(study['claims'])[['field','value','section','exact_quote']].head(50) if study['claims'] else pd.DataFrame())
for s in specs:
    print(s['identity']['tfl_id'], s['status'], 'Missing:', s['missing_required_fields'])
    for label in ('claims','review_claims'):
        print(label)
        display(pd.DataFrame(s[label])[['field','value','section','exact_quote','status']] if s[label] else pd.DataFrame())

## 5. Run regression tests
Synthetic claim and provenance tests; actual extraction quality still requires inspection.

In [ ]:
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_scoped_specification_phase3.py','tests/test_sap_scope_phase21.py','tests/test_sap_hybrid_phase2.py','tests/test_generic_library_phase1.py'],cwd=ROOT)

## 6. Export audit files
Run this cell even after an API/test error to preserve partial results. ZIP contains confidential source excerpts; do not share unless authorized.

In [ ]:
import shutil
from google.colab import files
ZIP_PATH = Path(shutil.make_archive(str(OUT), 'zip', OUT))
print('Local output:', OUT)
print('ZIP:', ZIP_PATH)
if SAVE_TO_DRIVE:
    destination = SAP_PATH.parent / 'Phase3_Outputs'
    destination.mkdir(parents=True, exist_ok=True)
    saved = Path(shutil.copy2(ZIP_PATH, destination / ZIP_PATH.name))
    print('Saved to Google Drive:', saved)
files.download(str(ZIP_PATH))